In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
pd.pandas.set_option("display.max_columns",None)


In [2]:
df = pd.read_csv("D:\Dataset\CAR DETAILS FROM CAR DEKHO.csv")

In [3]:
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


In [72]:
df['brand'] = df['name'].apply(lambda x: x.split()[0])

In [73]:
df.drop(columns=['name'],inplace=True, axis=1)

In [74]:
df.head()

,year,selling_price,km_driven,fuel,seller_type,transmission,owner,brand
0,2007,60000,70000,Petrol,Individual,Manual,First Owner,Maruti
1,2007,135000,50000,Petrol,Individual,Manual,First Owner,Maruti
2,2012,600000,100000,Diesel,Individual,Manual,First Owner,Hyundai
3,2017,250000,46000,Petrol,Individual,Manual,First Owner,Datsun
4,2014,450000,141000,Diesel,Individual,Manual,Second Owner,Honda


In [75]:
# Step 1: Identify unique classes
unique_classes = df['brand'].unique()

In [76]:
unique_classes

array(['Maruti', 'Hyundai', 'Datsun', 'Honda', 'Tata', 'Chevrolet',
       'Toyota', 'Jaguar', 'Mercedes-Benz', 'Audi', 'Skoda', 'Jeep',
       'BMW', 'Mahindra', 'Ford', 'Nissan', 'Renault', 'Fiat',
       'Volkswagen', 'Volvo', 'Mitsubishi', 'Land', 'Daewoo', 'MG',
       'Force', 'Isuzu', 'OpelCorsa', 'Ambassador', 'Kia'], dtype=object)

In [77]:
# Step 2: Create a mapping
class_mapping = {label: idx for idx, label in enumerate(unique_classes)}

In [78]:
# Step 3: Apply the mapping to the 'brand' column
df['brand_numeric'] = df['brand'].map(class_mapping)

In [79]:
# Step 4: Filter the dataset to include only classes ≤ 26
df = df[df['brand_numeric'] <= 26]

In [80]:
# Step 5: Separate features and target
X = df.drop(['brand', 'brand_numeric'], axis=1)
y_numeric = df['brand_numeric']

In [81]:
class_mapping

{'Maruti': 0,
 'Hyundai': 1,
 'Datsun': 2,
 'Honda': 3,
 'Tata': 4,
 'Chevrolet': 5,
 'Toyota': 6,
 'Jaguar': 7,
 'Mercedes-Benz': 8,
 'Audi': 9,
 'Skoda': 10,
 'Jeep': 11,
 'BMW': 12,
 'Mahindra': 13,
 'Ford': 14,
 'Nissan': 15,
 'Renault': 16,
 'Fiat': 17,
 'Volkswagen': 18,
 'Volvo': 19,
 'Mitsubishi': 20,
 'Land': 21,
 'Daewoo': 22,
 'MG': 23,
 'Force': 24,
 'Isuzu': 25,
 'OpelCorsa': 26,
 'Ambassador': 27,
 'Kia': 28}

In [82]:
df.head()

,year,selling_price,km_driven,fuel,seller_type,transmission,owner,brand,brand_numeric
0,2007,60000,70000,Petrol,Individual,Manual,First Owner,Maruti,0
1,2007,135000,50000,Petrol,Individual,Manual,First Owner,Maruti,0
2,2012,600000,100000,Diesel,Individual,Manual,First Owner,Hyundai,1
3,2017,250000,46000,Petrol,Individual,Manual,First Owner,Datsun,2
4,2014,450000,141000,Diesel,Individual,Manual,Second Owner,Honda,3


In [83]:
num_features = [feature for feature in df.columns if df[feature].dtype != 'O']
print('Num of Numerical Features :', len(num_features))

Num of Numerical Features : 4


In [84]:
cat_features = [feature for feature in df.columns if df[feature].dtype == 'O']
print('Num of Categorical Features :', len(cat_features))

Num of Categorical Features : 5


In [85]:
discrete_features=[feature for feature in num_features if len(df[feature].unique())<=25]
print('Num of Discrete Features :',len(discrete_features))

Num of Discrete Features : 0


In [86]:
continuous_features=[feature for feature in num_features if feature not in discrete_features]
print('Num of Continuous Features :',len(continuous_features))

Num of Continuous Features : 4


In [87]:
##these are the features with nan value
features_with_na=[features for features in df.columns if df[features].isnull().sum()>=1]
for feature in features_with_na:
    print(feature,np.round(df[feature].isnull().mean()*100,5), '% missing values')

In [88]:
features_with_na

[]

In [89]:
num_features = list(X.select_dtypes(exclude="object").columns)

In [90]:
num_features

['year', 'selling_price', 'km_driven']

In [91]:
df.head()

,year,selling_price,km_driven,fuel,seller_type,transmission,owner,brand,brand_numeric
0,2007,60000,70000,Petrol,Individual,Manual,First Owner,Maruti,0
1,2007,135000,50000,Petrol,Individual,Manual,First Owner,Maruti,0
2,2012,600000,100000,Diesel,Individual,Manual,First Owner,Hyundai,1
3,2017,250000,46000,Petrol,Individual,Manual,First Owner,Datsun,2
4,2014,450000,141000,Diesel,Individual,Manual,Second Owner,Honda,3


In [92]:
from sklearn.preprocessing import PowerTransformer
pt = PowerTransformer(method='yeo-johnson')
transform_features = ['km_driven', 'selling_price']
X_copy = pt.fit_transform(X[transform_features])

In [93]:
# Create Column Transformer with 3 types of transformers
or_columns = ['owner','seller_type']
oh_columns = ['fuel','transmission']
transform_columns= ['km_driven']

from sklearn.preprocessing import OneHotEncoder, StandardScaler,OrdinalEncoder, PowerTransformer
from sklearn.compose import ColumnTransformer 
from sklearn.pipeline import Pipeline

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder()
ordinal_encoder = OrdinalEncoder()

transform_pipe = Pipeline(steps=[
    ('transformer', PowerTransformer(method='yeo-johnson'))
])

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, oh_columns),
        ("Ordinal_Encoder", ordinal_encoder, or_columns),
        ("Transformer", transform_pipe, transform_columns),
        ("StandardScaler", numeric_transformer, num_features)
    ]
)

In [94]:
X = preprocessor.fit_transform(X)

In [95]:

X

array([[ 0.        ,  0.        ,  0.        , ..., -1.44819743,
        -0.76780464,  0.08086912],
       [ 0.        ,  0.        ,  0.        , ..., -1.44819743,
        -0.6381788 , -0.34786735],
       [ 0.        ,  1.        ,  0.        , ..., -0.26036426,
         0.16550144,  0.72397383],
       ...,
       [ 0.        ,  0.        ,  0.        , ..., -0.97306416,
        -0.68138741,  0.35954783],
       [ 0.        ,  1.        ,  0.        , ...,  0.68990227,
         0.62351276,  0.5096056 ],
       [ 0.        ,  0.        ,  0.        , ...,  0.68990227,
        -0.48262778, -0.56223559]])

In [96]:
df['owner'].value_counts()

owner
First Owner             2831
Second Owner            1103
Third Owner              304
Fourth & Above Owner      80
Test Drive Car            17
Name: count, dtype: int64

In [97]:
from sklearn.model_selection import  train_test_split
# separate dataset into train and test
X_train, X_test, y_train, y_test = train_test_split(X,y_numeric,test_size=0.3,random_state=42)
X_train.shape, X_test.shape

((3034, 13), (1301, 13))

In [98]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report,ConfusionMatrixDisplay, \
                            precision_score, recall_score, f1_score, roc_auc_score,roc_curve 
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [99]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

from sklearn.preprocessing import label_binarize

def evaluate_clf(true, predicted):
    # Calculate Accuracy
    acc = accuracy_score(true, predicted)
    
    # Handle multiclass classification for F1, Precision, and Recall
    f1 = f1_score(true, predicted, average='weighted')  # Use 'weighted' for multiclass
    precision = precision_score(true, predicted, average='weighted')  # Use 'weighted' for multiclass
    recall = recall_score(true, predicted, average='weighted')  # Use 'weighted' for multiclass
    
    # ROC AUC Score: Handle multiclass using one-vs-rest strategy
    if len(set(true)) > 2:  # Multiclass case
        true_binarized = label_binarize(true, classes=list(set(true)))
        predicted_binarized = label_binarize(predicted, classes=list(set(true)))
        roc_auc = roc_auc_score(true_binarized, predicted_binarized, average='weighted', multi_class='ovr')
    else:  # Binary case
        roc_auc = roc_auc_score(true, predicted)
    
    return acc, f1, precision, recall, roc_auc


In [100]:
models = {
    "Random Forest": RandomForestClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "Logistic Regression": LogisticRegression(),
     "K-Neighbors Classifier": KNeighborsClassifier(),
    "XGBClassifier": XGBClassifier(), 
     "CatBoosting Classifier": CatBoostClassifier(verbose=False),
     "Support Vector Classifier": SVC(),
    "AdaBoost Classifier": AdaBoostClassifier()

}

In [101]:
# Updated evaluate_models function
def evaluate_models(X, y, models):
    # Split dataset into train and test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    models_list = []
    accuracy_list = []
    auc = []
    
    for i in range(len(list(models))):
        model = list(models.values())[i]
        model.fit(X_train, y_train)  # Train model

        # Make predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        # Training set performance
        model_train_accuracy, model_train_f1, model_train_precision, \
        model_train_recall, model_train_rocauc_score = evaluate_clf(y_train, y_train_pred)

        # Test set performance
        model_test_accuracy, model_test_f1, model_test_precision, \
        model_test_recall, model_test_rocauc_score = evaluate_clf(y_test, y_test_pred)

        print(list(models.keys())[i])
        models_list.append(list(models.keys())[i])

        print('Model performance for Training set')
        print("- Accuracy: {:.4f}".format(model_train_accuracy))
        print('- F1 score: {:.4f}'.format(model_train_f1)) 
        print('- Precision: {:.4f}'.format(model_train_precision))
        print('- Recall: {:.4f}'.format(model_train_recall))
        if model_train_rocauc_score is not None:
            print('- Roc Auc Score: {:.4f}'.format(model_train_rocauc_score))
        else:
            print('- Roc Auc Score: Not applicable for multiclass classification')

        print('----------------------------------')

        print('Model performance for Test set')
        print('- Accuracy: {:.4f}'.format(model_test_accuracy))
        accuracy_list.append(model_test_accuracy)
        print('- F1 score: {:.4f}'.format(model_test_f1))
        print('- Precision: {:.4f}'.format(model_test_precision))
        print('- Recall: {:.4f}'.format(model_test_recall))
        if model_test_rocauc_score is not None:
            print('- Roc Auc Score: {:.4f}'.format(model_test_rocauc_score))
            auc.append(model_test_rocauc_score)
        else:
            print('- Roc Auc Score: Not applicable for multiclass classification')
            auc.append(None)
        print('='*35)
        print('\n')
        
    report = pd.DataFrame(list(zip(models_list, accuracy_list)), columns=['Model Name', 'Accuracy']).sort_values(by=['Accuracy'], ascending=False)
        
    return report


In [102]:
base_model_report =evaluate_models(X=X, y=y_numeric, models=models)

Random Forest
Model performance for Training set
- Accuracy: 0.9893
- F1 score: 0.9893
- Precision: 0.9894
- Recall: 0.9893
- Roc Auc Score: 0.9936
----------------------------------
Model performance for Test set
- Accuracy: 0.5098
- F1 score: 0.5005
- Precision: 0.5007
- Recall: 0.5098
- Roc Auc Score: 0.7041


Decision Tree
Model performance for Training set
- Accuracy: 0.9893
- F1 score: 0.9893
- Precision: 0.9894
- Recall: 0.9893
- Roc Auc Score: 0.9933
----------------------------------
Model performance for Test set
- Accuracy: 0.4694
- F1 score: 0.4733
- Precision: 0.4797
- Recall: 0.4694
- Roc Auc Score: 0.6887


Gradient Boosting
Model performance for Training set
- Accuracy: 0.6416
- F1 score: 0.6344
- Precision: 0.6741
- Recall: 0.6416
- Roc Auc Score: 0.7679
----------------------------------
Model performance for Test set
- Accuracy: 0.4660
- F1 score: 0.4452
- Precision: 0.4652
- Recall: 0.4660
- Roc Auc Score: 0.6650


Logistic Regression
Model performance for Training 